# ランダムな整数の衝突

UUID バージョン 4 {footcite:p}`Leach2005` のようにランダムで十分長い整数は、
同じ整数が選択される（以降、「衝突する」）確率が十分低いため、
単一のプロセスでなく複数のプロセスやコンピュータ上で生成される ID などで
使用されることがある。
ここでは、そのような乱数が衝突する確率が実際どの程度になるのか計算してみる。

In [ ]:
from til_utils.plot_common import load_common_config

load_common_config()

## 衝突する確率

まず、衝突する確率の計算式を考える。
整数を $0$ から $m-1$ までの $m$ 個の中から
$n$ 回選択する場合、
衝突する確率は、

\begin{align*}
p(m, n) &= 1 -
    \frac{m    }{m} \cdot
    \frac{m-1  }{m} \cdot
    \frac{m-2  }{m} \cdot
    \frac{m-3  }{m} \cdot \cdots \cdot
    \frac{m-n+1}{m}
\\
    &= 1 - \frac{m!}{(m-n)! \ m^n}
\end{align*}

のように書ける。

## 計算

PC で計算しやすいように計算式を変形する。
まず、階乗よりもライブラリでの計算が容易なガンマ関数を用いて式を変形する。

\begin{align*}
    p(m, n) &= 1 - \frac{m!}{(m-n)! \ m^n}
\\
    &= 1 - \frac{\Gamma(m+1)}{\Gamma(m-n+1) \ m^n}
\end{align*}

ガンマ関数は発散しやすいため、ガンマ関数の対数を用いるように式を変形する。

\begin{align*}
    \log(1 - p(m, n)) &= \log \frac{\Gamma(m+1)}{\Gamma(m-n+1) \ m^n}
\\
    &= \log \Gamma(m+1) - \log \Gamma(m-n+1) - n \log m
\end{align*}

最後に $p(m, n)$ について解くと以下になる。

\begin{align*}
    p(m, n) &= 1 - \exp\left(\log \Gamma(m+1) - \log \Gamma(m-n+1) - n \log m\right)
\end{align*}

これをもとにして確率の計算を行うと以下のようになる。

In [ ]:
import mpmath
import plotly.express as px
from til_utils.plot_common import show_figure

mpmath.mp.dps = 200

def calc_random_number_collision_rate(
    num_selectable_numbers: int, num_selected_numbers: int
) -> float:
    """ランダムな整数が衝突する確率を計算する。

    Args:
        num_selectable_numbers (int): 選択肢の数。
        num_selected_numbers (int): 選択する数。

    Returns:
        float: 衝突する確率。
    """

    if num_selected_numbers > num_selectable_numbers:
        # 確実に衝突するため 1 とする。
        return 1.0

    # 差が少ない数同士の引き算になるようで、
    # 打ち切り誤差により float では計算しきれなくなるため、
    # mpmath で桁数多めに計算する。
    log_prob_no_collision = (
        mpmath.loggamma(num_selectable_numbers + 1)
        - mpmath.loggamma(num_selectable_numbers - num_selected_numbers + 1)
        - num_selected_numbers * mpmath.log(num_selectable_numbers)
    )

    return float(-mpmath.expm1(log_prob_no_collision))


num_bits_vec = []
num_selected_numbers_vec = []
rate_vec = []
num_bits_list = [8, 16, 32, 64, 122, 128]
num_selected_numbers_list = []
for i in range(1, 13):
    base = 10 ** i
    num_selected_numbers_list = num_selected_numbers_list + [2 * base, 5 * base, 10 * base]
for num_bits in num_bits_list:
    for num_selected_numbers in num_selected_numbers_list:
        num_selectable_numbers = 2**num_bits

        rate = calc_random_number_collision_rate(
            num_selectable_numbers=num_selectable_numbers,
            num_selected_numbers=num_selected_numbers,
        )

        num_bits_vec.append(num_bits)
        num_selected_numbers_vec.append(num_selected_numbers)
        rate_vec.append(rate)

fig = px.line(
    x=num_selected_numbers_vec,
    y=rate_vec,
    color=num_bits_vec,
    log_x=True,
    log_y=True,
)
fig.update_layout(
    legend_title_text="選択する整数のビット数",
    xaxis_title="選択した整数の数",
    yaxis_title="確率",
    title="ランダムに選択した整数が衝突する確率",
    xaxis={
        "exponentformat": "power",
    },
    yaxis={
        "exponentformat": "power",
    },
)
show_figure(fig)

```{hint}
122 ビットは UUID バージョン 4 の場合にあたる。
```

## 近似式

選択する整数の数 $n$ が小さくない場合について近似を考える。
$m \gg n \gg 1$ とし、
マクローリン展開

$$
\log(1 - x) = - \sum_{k=1}^\infty \frac{x^k}{k} \approx -x
$$

を用いると、

\begin{align*}
\sum_{i=0}^{n-1} \log\left(1 - \frac{i}{m} \right) & \approx
    - \sum_{i=0}^{n-1} \frac{i}{m}
\\
    & = - \frac{n(n-1)}{2m}
\\
    & \approx - \frac{n^2}{2m}
\end{align*}

となるから、

$$
p(m, n) \approx 1 - \exp\left(- \frac{n^2}{2m} \right)
$$

のように近似できる。

## まとめ

ランダムな整数が衝突する確率を計算してグラフにすることができた。
ランダムな整数で ID などを決定する際に参考にしたい。


## 参考

```{footbibliography}
```